# Ch2. 신경망 — MNIST

머신러닝계의 Hello World. 흑백 손글씨 숫자 이미지를 분류한다.

- 28×28 픽셀, 0~9 범주
- 1980년대 NIST 데이터. 훈련 6만 장 + 테스트 1만 장
- 성능 지표: **정확도** = 맞힌 비율 (true class 비율)

필기에서 `tr_img`, `tr_lbl`로 줄여 쓴 이름을 여기서는 `train_images`, `train_labels`로 둔다. 같은 값이다.

용어

| 말 | 뜻 |
|----|----|
| class | 분류 문제의 category. 여기선 0~9 |
| sample | 데이터 포인트 하나 (이미지 한 장) |
| label | 그 sample의 class |

`(참고)` NumPy는 GPU와 autodifferentiation을 지원하지 않는다. 그래서 신경망 학습 자체는 Keras가 맡는다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

print("train", train_images.shape, train_labels.shape)
print("test ", test_images.shape, test_labels.shape)
print("dtype", train_images.dtype, "label 예시", train_labels[:10])

# 뒤에서 reshape 하기 전에, 슬라이싱·그림용으로 원본을 남겨 둔다.
images_2d = train_images.copy()
labels_2d = train_labels.copy()


## data.shape — 차원의 크기

- `(404, 13)` → 속성 13개짜리 데이터 404개
- `(60000, 28, 28)` → `[28, 28]`짜리 데이터 60000개

총 70,000장. 훈련 60,000 + 테스트 10,000.


In [ ]:
print("ex1 느낌의 shape 예시", np.zeros((404, 13)).shape)
print("ex2 train_images.shape", train_images.shape)
print("한 장", train_images[0].shape, "라벨", train_labels[0])


## 모델 정의와 컴파일

`Sequential`은 데이터 흐름처럼 층을 순서대로 쌓는다. 여기서는 층이 2개.

- 첫 `Dense(512, relu)`: 512개 유닛. 이전 층의 출력을 다음 층 입력으로 보낸다.
- 둘째 `Dense(10, softmax)`: 0~9, 10개 클래스 안에서 확률을 내는 출력층.
- 활성화 함수: 이전 층 값을 어떻게 다음 층으로 보낼지 정한다.

`compile`은 학습 방법·규칙을 정한다.

| 인자 | 질문 | 여기 값 |
|------|------|---------|
| optimizer | 가중치를 어떤 방식으로 업데이트? | `adam` |
| loss | 손실함수. 틀린 정도 | `sparse_categorical_crossentropy` |
| metrics | 평가 지표 | `accuracy` |

라벨이 정수(0~9)라서 `sparse_` 크로스엔트로피를 쓴다. 원-핫이면 `categorical_crossentropy`다.


In [ ]:
model = keras.Sequential([
    keras.Input(shape=(784,)),
    layers.Dense(512, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 전처리

원본은 `(60000, 28, 28)`, 픽셀 값 0~255 (`uint8`).

1. `reshape((60000, 784))` — 1차원으로 편다. 이 MLP는 1차원 입력만 받는다. 28×28 = 784.
2. `astype("float32") / 255` — 0~1로 바꾼다. 숫자 스케일을 맞춰 학습을 단순하게 만든다.

테스트 이미지도 **같은 변환**을 해야 `evaluate` / `predict`가 맞는다. 교재 필기에는 훈련만 적혀 있지만, 테스트에 안 하면 shape 오류가 난다.


In [ ]:
train_images = train_images.reshape((60000, 784)).astype("float32") / 255
test_images = test_images.reshape((10000, 784)).astype("float32") / 255

print(train_images.shape, train_images.dtype, float(train_images.min()), float(train_images.max()))
print(test_images.shape)


## model.fit — epochs, batch_size

- `batch_size=128`: 128장씩 묶어서 한 스텝.
- `epochs=5`: 전체 훈련 데이터를 5바퀴.
- 너무 적으면 과소적합, 너무 많으면 과대적합.

예) 훈련 60,000장 / 128 ≈ 469스텝이 1에포크. 5에포크면 그 과정을 5번.


In [ ]:
history = model.fit(
    train_images,
    train_labels,
    epochs=5,
    batch_size=128,
)


## 예측

`test_images`에서 10장만 뽑아 예측한다. 결과는 샘플마다 길이 10인 확률 벡터다. (클래스 0~9)

- `predictions[n]` : n번째 샘플의 확률 10개
- `predictions[n].argmax()` : 그 중 가장 큰 칸의 인덱스 = 모델이 고른 숫자


In [ ]:
n = 0
predictions = model.predict(test_images[0:10])

print("predictions.shape", predictions.shape)  # (10, 10)
print("0번 확률", np.round(predictions[n], 3))
print("0번 예측", predictions[n].argmax(), "정답", test_labels[n])

for i in range(10):
    print(i, "pred", predictions[i].argmax(), "true", test_labels[i])


## 손실과 정확도

`evaluate`는 테스트 전체에서 loss와 accuracy를 낸다.


In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print("test_loss", test_loss)
print("test_acc", test_acc)


## 신경망을 위한 데이터 표현 — 텐서

텐서는 데이터를 담는 컨테이너다. 축(axis)의 개수가 행렬로 일반화된 모습이고, `ndim`이 랭크다.

- 스칼라: 랭크 0. `np.array(12)`
- 벡터: 랭크 1. `shape (5,)` → `[1, 2, 3, 4, 5]`
- 행렬: 랭크 2. `shape (2, 5)`

MNIST 원본

- `images_2d.shape` → `(60000, 28, 28)` → `ndim` 3
- `dtype` → `uint8` (0~255). matplotlib가 그릴 수 있는 픽셀 값.

`plt.cm.binary`는 흑백 계열 컬러맵이다.


In [ ]:
scalar = np.array(12)
vector = np.array([12, 3, 6, 14])
matrix = np.array([[1, 2, 3, 4, 5],
                   [1, 2, 3, 4, 5]])

print("scalar", scalar.ndim, scalar.shape)
print("vector", vector.ndim, vector.shape)
print("matrix", matrix.ndim, matrix.shape)

print("MNIST", images_2d.shape, images_2d.ndim, images_2d.dtype)

plt.imshow(images_2d[4], cmap=plt.cm.binary)
plt.title("label = %d" % labels_2d[4])
plt.axis("off")
plt.show()


## 실제 데이터와 텐서 랭크

| 데이터 | rank | shape 축 |
|--------|------|----------|
| 벡터 (표 형태) | 2 | (samples, features) |
| 시계열 / 시퀀스 | 3 | (samples, timesteps, features) |
| 이미지 | 4 | (samples, height, width, channels) |
| 동영상 | 5 | (samples, frames, height, width, channels) |

MNIST를 흑백 채널까지 붙이면 `(60000, 28, 28, 1)`이라 rank 4가 된다. 지금 `load_data()` 결과는 채널 축이 빠져 rank 3이다.


In [ ]:
mnist_rank4 = images_2d.reshape(60000, 28, 28, 1)
print("rank3", images_2d.shape, images_2d.ndim)
print("rank4", mnist_rank4.shape, mnist_rank4.ndim)


## 텐서 슬라이싱

- `images[10:100]` → 10번~99번, 90장
- `images[:, 14:, 14:]` → 모든 이미지의 **오른쪽 아래** 14×14
- `images[:, 7:-7, 7:-7]` → 모든 이미지의 **중앙** 14×14


In [ ]:
print("90장", images_2d[10:100].shape)
print("우하단 14x14", images_2d[:, 14:, 14:].shape)
print("중앙 14x14", images_2d[:, 7:-7, 7:-7].shape)

fig, axes = plt.subplots(1, 3, figsize=(8, 2.6))
axes[0].imshow(images_2d[4], cmap=plt.cm.binary)
axes[0].set_title("원본")
axes[1].imshow(images_2d[4, 14:, 14:], cmap=plt.cm.binary)
axes[1].set_title("우하단")
axes[2].imshow(images_2d[4, 7:-7, 7:-7], cmap=plt.cm.binary)
axes[2].set_title("중앙")
for ax in axes:
    ax.axis("off")
plt.show()


## 배치

`batch = images[128 * n : 128 * (n + 1)]`

n번째 배치. 한 번에 128장.


In [ ]:
n = 3
batch = images_2d[128 * n : 128 * (n + 1)]
print("n =", n, "batch", batch.shape)


## 텐서 연산

원소별 덧셈 `z = x + y`는 NumPy/Keras가 C로 된 BLAS를 써서 빠르다. 파이썬 for 루프(`naive_add`)는 같은 일이라도 훨씬 느리다.

브로드캐스팅: `x`가 `(32, 10)`, `y`가 `(10,)`여도 `x + y`가 된다. y가 행마다 복제된다.

텐서 곱(행렬 내적)은 `z = x @ y`. 원소별 곱(`*`)과는 다르다.

reshape는 값을 바꾸지 않고 모양만 바꾼다.

```
x = [[0, 1],
     [2, 3],
     [4, 5]]
x.reshape((6, 1)) → 6행 1열
x.reshape((2, 3)) → [[0, 1, 2],
                     [3, 4, 5]]
```


In [ ]:
import time

def naive_add(x, y):
    out = np.empty_like(x)
    for i in range(x.shape[0]):
        for j in range(x.shape[1]):
            out[i, j] = x[i, j] + y[i, j]
    return out

rng = np.random.default_rng(0)
a = rng.random((400, 400)).astype("float32")
b = rng.random((400, 400)).astype("float32")

t0 = time.perf_counter()
z_fast = a + b
t_fast = time.perf_counter() - t0

t0 = time.perf_counter()
z_slow = naive_add(a, b)
t_slow = time.perf_counter() - t0

print("x + y     %.4f s" % t_fast)
print("naive_add %.4f s" % t_slow)
print("같나?", np.allclose(z_fast, z_slow))

# 브로드캐스팅
x = rng.random((32, 10))
y = rng.random((10,))
print("broadcast", (x + y).shape)

# 내적
x = rng.random((32, 10))
y = rng.random((10, 4))
print("dot", (x @ y).shape)

# reshape
x = np.array([[0, 1],
              [2, 3],
              [4, 5]])
print("원본", x.shape)
print(x.reshape((6, 1)))
print(x.reshape((2, 3)))


## 경사하강법 (Gradient Descent)

θ는 파라미터, α는 학습률.

`θ_{t+1} = θ_t - α ∇J(θ_t)`

쉽게 말하면, 기울기 반대 방향으로 조금 움직여 손실을 줄인다. `y = x²`이면 기울기는 `2x`이고, 최솟값은 x=0이다.

| 종류 | 특징 |
|------|------|
| Batch GD | 전체 데이터로 기울기. 느리지만 안정적 |
| Stochastic GD | 샘플 하나로 기울기. 빠르지만 노이즈가 많고 local minima에 흔들릴 수 있다 |
| Mini-batch GD | 묶음으로 계산. 지금 `fit`의 `batch_size=128`이 이 방식. 전체 최솟값은 아니고 국지적 최솟값으로 간다 |


In [ ]:
def gradient_descent_simple(lr=0.1, epochs=50):
    x = 5.0
    history = [x]
    for _ in range(epochs):
        grad = 2 * x          # d/dx (x^2) = 2x
        x = x - lr * grad
        history.append(x)
    return x, history

final_x, hist = gradient_descent_simple()
print("final_x", final_x)
print("앞 5스텝", [round(v, 4) for v in hist[:5]])
print("끝 5스텝", [round(v, 4) for v in hist[-5:]])


## 기울기가 0인 곳

파란 계단은 `x²` 위에서 기울기 반대 방향(최솟값)으로 내려가는 경로다.  
빨간 곡선은 울퉁불퉁한 손실의 스케치다. `J'(x) = 0`인 지점이 기울기가 0인 곳이고, 그 중에는 local minima도 있다.


In [ ]:
xs = np.linspace(-1, 6, 200)
plt.figure(figsize=(6, 3.2))
plt.plot(xs, xs ** 2, color="black", label="y = x^2")
plt.plot(hist, np.square(hist), "o-", color="royalblue", ms=3, label="GD 경로")
plt.scatter([0], [0], color="red", zorder=3, label="J'(x)=0 (최소)")
plt.xlabel("x")
plt.ylabel("y")
plt.title("y = x^2 에서 기울기 반대 방향")
plt.legend()
plt.tight_layout()
plt.show()

# 울퉁불퉁한 손실 스케치. 기울기 0인 점을 눈으로 표시.
t = np.linspace(-2, 6, 400)
J = np.sin(t) + 0.15 * (t - 1) ** 2
dJ = np.cos(t) + 0.3 * (t - 1)
zeros = np.where(np.diff(np.sign(dJ)))[0]

plt.figure(figsize=(6, 3.2))
plt.plot(t, J, color="black")
plt.scatter(t[zeros], J[zeros], color="red", label="J'(x)=0")
plt.title("local minima 가 있는 손실")
plt.legend()
plt.tight_layout()
plt.show()


## 기하학적 텐서 연산

1. **이동** — 점에 벡터를 더한다. `[x, y] + [a, b]`
2. **회전** — 회전 행렬을 곱한다.
3. **스케일** — 대각 행렬. x방향·y방향 배율이 다를 수 있다. (필기의 -0.5는 y를 뒤집으며 줄인다는 뜻)
4. **선형 변환** — 회전과 스케일을 행렬 하나로 합칠 수 있다.
5. **아핀 변환** — 선형 변환 뒤에 이동을 붙인다. `y = W @ x + b`
6. **ReLU** — `relu(x) = max(x, 0)`. 음수는 0으로 눌러 비선형을 만든다. 모델 첫 층의 `activation="relu"`가 이 함수다.


In [ ]:
point = np.array([1.0, 0.6])
move = np.array([0.8, 0.3])
translated = point + move

theta = np.deg2rad(40)
rotation = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)],
])
rotated = rotation @ point

scale = np.array([
    [1.4, 0.0],
    [0.0, -0.5],   # y 방향은 뒤집히며 절반
])
scaled = scale @ point

# 선형 변환 = 회전 행렬과 스케일 행렬의 곱
W = rotation @ scale
b = move
affine = W @ point + b

def relu(x):
    return np.maximum(x, 0)

grid = np.array([[0.2, 0.2],
                 [1.2, 0.3],
                 [0.4, 1.1],
                 [1.3, 1.0],
                 [-0.6, 0.8],
                 [-0.4, -0.5]])

def apply_affine(pts):
    return (pts @ W.T) + b

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].scatter(grid[:, 0], grid[:, 1], c="black")
axes[0].set_title("원본")
moved = grid + move
axes[1].scatter(moved[:, 0], moved[:, 1], c="black")
axes[1].set_title("이동")
aff = apply_affine(grid)
axes[2].scatter(aff[:, 0], aff[:, 1], c="black")
axes[2].set_title("아핀  y = Wx + b")
for ax in axes:
    ax.set_aspect("equal", adjustable="datalim")
    ax.axhline(0, color="gray", lw=0.5)
    ax.axvline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

xs = np.linspace(-2, 2, 200)
plt.figure(figsize=(4.5, 2.6))
plt.plot(xs, relu(xs), color="black")
plt.axhline(0, color="gray", lw=0.5)
plt.axvline(0, color="gray", lw=0.5)
plt.title("relu(x) = max(x, 0)")
plt.tight_layout()
plt.show()

print("point     ", point)
print("translated", translated)
print("rotated   ", np.round(rotated, 3))
print("scaled    ", scaled)
print("affine    ", np.round(affine, 3))
print("relu      ", relu(np.array([-1.5, 0.0, 2.0])))


## 한 줄로 다시 보기

1. MNIST를 읽고 shape를 확인한다. `(60000, 28, 28)`, `uint8`.
2. `Dense(512, relu)` → `Dense(10, softmax)` 를 쌓고, adam + sparse categorical crossentropy로 컴파일한다.
3. 784로 펴고 255로 나눠 0~1로 만든 뒤, `epochs=5`, `batch_size=128`로 학습한다.
4. `predict`는 클래스별 확률, `argmax`가 예측 숫자, `evaluate`가 테스트 정확도다.
5. 슬라이싱·배치·브로드캐스팅·`@`·`reshape`는 그 데이터를 다루는 텐서 연산이다.
6. 학습이 가중치를 옮기는 방식은 경사하강, 한 층의 비선형은 ReLU (`max(x, 0)`) 다.
